# Activity: Debug our Fibonacci Calculation
In this activity, we'll continue the development of our Fibonacci sequence calculations. Previously, we constructed an empty `MyFibonacciSequenceModel` struct instance, by calling the default `MyFibonacciSequenceModel()` constructor method. We then passed this model instance to the `fibonacci!(...)` method to compute the Fibonacci numbers for a given index. This worked fine, but we can do better!

We'll formulate a `build(...)` method which is responsible for properly constructing  `MyFibonacciSequenceModel` instances, and we'll make sure the `fibonacci!(...)` method is robust to bad inputs, to ensure it works correctly, and responds gracefully to errors.

### Review
 A [Fibonacci sequence](https://en.wikipedia.org/wiki/Fibonacci_sequence) is composed of the Fibonacci numbers $F_{n}$ where:
$$
\begin{align*}
F_{0} & = 0 \quad n = 0\\
F_{1} & = 1 \quad n = 1\\
F_{n} & = F_{n-2} + F_{n-1}\quad{n\geq{2}}
\end{align*}
$$

Let's implement a `build(...)` method to create a `MyFibonacciSequenceModel` instance with a specified size, and default values for the other arguments. We'll then pass this instance to the `fibonacci!(...)` method to compute the Fibonacci numbers for a given index.

Let's go!

___


## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.
* The [include command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 
* In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), check out that documentation for more information on the functions and types used in this material.

In [1]:
include("Include.jl"); # load the Include.jl file

### Types and Functions
Let's define the `MyFibonacciSequenceModel` type, and some no data iteration tagging types (which tells us which iteration method to use). 

In [2]:
abstract type AbstractSequenceModel end # This is an abstract type for sequence models
abstract type AbstractIterationModel end # This is an abstract type for iteration models

"""
    MyFibonacciSequenceModel <: AbstractSequenceModel

A mutable struct to represent a Fibonacci sequence. 

### Fields
- `n::Int64`: The number of elements in the sequence.
- `sequence::Dict{Int64, Int64}`: The sequence itself, stored as a dictionary with indices as keys and Fibonacci numbers as values.

"""
mutable struct MyFibonacciSequenceModel <: AbstractSequenceModel

    # data -
    n::Int64 # number of elements in the sequence
    sequence::Dict{Int64, Int64} # the sequence itself

    # constructor -
    MyFibonacciSequenceModel() = new();
end

"""
    MyForLoopIterationModel <: AbstractIterationModel

An immutable struct to represent a for loop iteration model. This type has no fields and serves as a marker for a for loop iteration implementation.
"""
struct MyForLoopIterationModel <: AbstractIterationModel
    MyForLoopIterationModel() = new();
end

"""
    MyWhileLoopIterationModel <: AbstractIterationModel

An immutable struct to represent a while loop iteration model. This type has no fields and serves as a marker for a while loop iteration implementation.
"""
struct MyWhileLoopIterationModel <: AbstractIterationModel
    MyWhileLoopIterationModel() = new();
end

MyWhileLoopIterationModel

The `fibonacci!(...)` method will compute the Fibonacci numbers for a given index, given an instance of `MyFibonacciSequenceModel`. Remember, the `!` at the end of the method name indicates that this method will modify the input model instance in place.
* _Mutating methods_: In Julia, a `!` at the end of a function name indicates that the function _modifies its arguments_ in some way that will be visible after the method execution has ended. In this case, the `fibonacci!` method modifies the `my_sequence_model` by populating its `sequence` field with Fibonacci numbers.
* _Is there something magical about the `!` character_? No adding the `!` character to the end of function name is _not magic_. It's just a convention to help identify functions that may change state or data outide the local scope of the function. In this particular case, the `fibonacci!` method modifies the `my_sequence_model` by populating its `sequence` field with Fibonacci numbers.
* _Optional keyword arguments_: The `iterationmodel::T` optional argument defaults to an instance of `MyForLoopIterationModel`, thus, we'll use the for-loop iteration method by default. If we wanted to use our while loop implementation, should we pass in a `MyWhileLoopIterationModel` instance.

In [ ]:
# -- PRIVATE METHODS BELOW HERE ------------------------------------------------------------------------------------------------------- #
function _fibonacci(sequencemodel::MyFibonacciSequenceModel, iterationmodel::MyForLoopIterationModel)

    @info "Debug message: We are using the for loop iteration model"

    # initialize -
    n = sequencemodel.n;
    sequence = Dict{Int64, Int64}();

    # we know the first two elements -
    sequence[0] = 0;
    sequence[1] = 1;

    # main loop, compute F₂, ....
    for i ∈ 2:n # what is this short-hand for?
        sequence[i] = sequence[i-1] + sequence[i-2]
    end

    # update the model -
    sequencemodel.sequence = sequence;
end

function _fibonacci(sequencemodel::MyFibonacciSequenceModel, iterationmodel::MyWhileLoopIterationModel)

    @info "Debug message: We are using the while loop iteration model"

    # check: is n legit?
    n = sequencemodel.n;
    sequence = Dict{Int64, Int64}();
    
    # main loop 
    should_loop_continue = true
    i = 0;
    while (should_loop_continue == true)
       
        # conditional logic: hardcode 0, 1 else gets all other cases
        if (i == 0)
            sequence[i] = 0; 
        elseif (i == 1)
            sequence[i] = 1;
        else
            sequence[i] = sequence[i - 1] + sequence[i - 2]
        end

        # update i -
        i += 1; # this is short-hand for i = i + 1

        # check: should we go around again?
        if (i>n)
            should_loop_continue = false;
        end
    end
    
    # update the model -
    sequencemodel.sequence = sequence;
end
# -- PRIVATE METHODS ABOVE HERE ------------------------------------------------------------------------------------------------------- #

# -- PUBLIC METHODS BELOW HERE -------------------------------------------------------------------------------------------------------- #
"""
    function fibonacci!(sequencemodel::MyFibonacciSequenceModel; 
        iterationmodel::T = MyForLoopIterationModel()) where T <: AbstractIterationModel

This function computes the Fibonacci sequence, given sequence and iteration models. 
The sequence model is updated in place (the sequence model is mutable). 
The iteration model is used to determine the type of loop to use.

# Arguments
- `sequencemodel::MyFibonacciSequenceModel`: The sequence model to update. The sequence model must have a field `n::Int64` that is the number of elements to compute.
- `iterationmodel::T`: The iteration model to use. It must be a subtype of `AbstractIterationModel`. The default is `MyForLoopIterationModel`.

There is no return value. The `sequencemodel` is updated in place.
"""
function fibonacci!(sequencemodel::MyFibonacciSequenceModel; 
    iterationmodel::T = MyForLoopIterationModel()) where T <: AbstractIterationModel
    
    # Status: If we get here, then we know n >= 0
    _fibonacci(sequencemodel, iterationmodel); # multiple dispatch to the appropriate implementation
end;

## Task 1: Create a build method for MyFibonacciSequenceModel
In this task, we will create a `build(...)` method that constructs a `MyFibonacciSequenceModel` instance with a specified size and default values for the other fields. This method will ensure that the model is properly initialized before being passed to the `fibonacci!(...)` method.

__Requirements__:
* _Arguments_: The `build(...)` method will takes the type of thing we want to build, i.e., `MyFibonacciSequenceModel` , the sequence size `n::Int` and the default value `defaultvalue::Int` parameters will be passed in a `data::NamedTuple` instance. The `build(...)` method returns a properly constructed `MyFibonacciSequenceModel` instance.
* _Error conditions_: Fill me in

Ready, set go!

In [ ]:
function build(modeltype::Type{MyFibonacciSequenceModel}, data::NamedTuple)::MyFibonacciSequenceModel
    
    # build a new model -
    sequencemodel = modeltype();
    
     
    
    
    # return the model -
    return sequencemodel; 
end